In [ ]:
# Stage-2 step-200 CPU pilot evidence audit. Runtime -> Run all. CPU High-RAM recommended.
from pathlib import Path
import os, re, subprocess
REPOSITORY_URL = 'https://github.com/GuillermoTafoya/MRIxFields.git'
TRAINING_EVIDENCE_COMMIT = '82633d66e5ea47f96b149ea22cc192fcf4526f06'
AUDIT_IMPLEMENTATION_COMMIT = '0f50b162cc9788699ae1fe152da6273e1911ba51'
if re.fullmatch(r'[0-9a-f]{40}', AUDIT_IMPLEMENTATION_COMMIT) is None:
    raise RuntimeError('Notebook is unsealed: audit implementation commit is not pinned.')
REPO_DIR = Path('/content/MRIxFields-stage2-step200-audit-v9-' + AUDIT_IMPLEMENTATION_COMMIT[:12])
def git_probe(repo_dir, *args):
    env = os.environ.copy(); env['GIT_OPTIONAL_LOCKS'] = '0'
    return subprocess.run(['git', *args], cwd=repo_dir, text=True, capture_output=True, env=env)
def validate_checkout(repo_dir):
    repo_dir = Path(repo_dir)
    if not repo_dir.is_dir(): raise RuntimeError('Existing audit checkout is not a directory.')
    def text(*args):
        result = git_probe(repo_dir, *args)
        if result.returncode: raise RuntimeError({'read_only_git_check_failed': args, 'returncode': result.returncode})
        return result.stdout.strip()
    if text('rev-parse', '--is-inside-work-tree') != 'true': raise RuntimeError('Existing path is not a Git worktree.')
    if Path(text('rev-parse', '--show-toplevel')).resolve() != repo_dir.resolve(): raise RuntimeError('Unexpected checkout root.')
    if text('remote').splitlines() != ['origin'] or text('remote', 'get-url', '--all', 'origin').splitlines() != [REPOSITORY_URL]: raise RuntimeError('Audit checkout origin changed.')
    if text('rev-parse', 'HEAD') != AUDIT_IMPLEMENTATION_COMMIT: raise RuntimeError('Existing audit checkout is at the wrong commit.')
    symbolic = git_probe(repo_dir, 'symbolic-ref', '-q', 'HEAD')
    if symbolic.returncode != 1 or symbolic.stdout.strip(): raise RuntimeError('Audit checkout must be detached.')
    if text('status', '--porcelain=v1', '--untracked-files=all'): raise RuntimeError('Existing audit checkout is dirty.')
    if git_probe(repo_dir, 'merge-base', '--is-ancestor', TRAINING_EVIDENCE_COMMIT, 'HEAD').returncode: raise RuntimeError('Audit commit does not descend from training evidence.')
    changed = text('diff', '--name-only', TRAINING_EVIDENCE_COMMIT, 'HEAD').splitlines()
    allowed_exact = {'src/fieldbridge/evaluation/stage2_unified_gate01_p0006.py', 'src/fieldbridge/evaluation/stage2_unified_preflight.py', 'src/fieldbridge/evaluation/stage2_step200_pilot_audit.py', 'src/fieldbridge/evaluation/stage2_step200_inference_audit.py'}
    allowed_prefixes = ('notebooks/', 'tests/', 'docs/')
    disallowed = [path for path in changed if path not in allowed_exact and not path.startswith(allowed_prefixes)]
    if disallowed: raise RuntimeError({'audit_diff_outside_restricted_scope': disallowed})
    protected = ('src/fieldbridge/models/', 'src/fieldbridge/training/', 'src/fieldbridge/data/', 'configs/', 'pyproject.toml')
    if any(path == 'pyproject.toml' or path.startswith(protected[:-1]) for path in changed): raise RuntimeError('Protected training/model/data/config/package objects changed.')
    return {'checkout_reused_without_mutation': True, 'changed_path_count': len(changed)}
checkout_reused = REPO_DIR.exists()
if checkout_reused:
    checkout_state = validate_checkout(REPO_DIR)
else:
    subprocess.run(['git', 'clone', '--no-checkout', REPOSITORY_URL, str(REPO_DIR)], check=True)
    subprocess.run(['git', 'fetch', 'origin', AUDIT_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '--detach', AUDIT_IMPLEMENTATION_COMMIT], cwd=REPO_DIR, check=True)
    checkout_state = validate_checkout(REPO_DIR); checkout_state['checkout_reused_without_mutation'] = False
def git_text(*args):
    result = git_probe(REPO_DIR, *args)
    if result.returncode: raise RuntimeError({'read_only_git_check_failed': args, 'returncode': result.returncode})
    return result.stdout.strip()
print({'audit_implementation_commit': AUDIT_IMPLEMENTATION_COMMIT, 'training_evidence_commit': TRAINING_EVIDENCE_COMMIT, 'detached_clean_checkout': True, 'training_critical_objects_unchanged': True, 'checkout_reused_without_mutation': checkout_state['checkout_reused_without_mutation'], 'runtime_recommendation': 'CPU High-RAM', 'gpu_used': False, 'packages_installed_or_downloaded': False}, flush=True)
operator = REPO_DIR / 'notebooks/stage2_step200_pilot_audit_operator.py'
exec(compile(operator.read_text(encoding='utf-8'), str(operator), 'exec'), globals())
